# Import functions

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
%run "DataHelpers.ipynb"

# Run all models against all non-PCA feature selections

In [2]:
for modelName in ModelVariant.__members__:
    
    print('\n\n')
    print(f'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII')
    print(f'*** - Applying {modelName} to features - Start')
    print(f'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\n')

    total = len(FeatureVariant)
    counter = 1
    
    for GENE_FILE_VARIANT in FeatureVariant:
        if GENE_FILE_VARIANT == FeatureVariant.AUTOMATED:
            continue

        print(f'\n=========================================================')
        print(f"======= {counter}/{total} - FeatureSet {GENE_FILE_VARIANT} - Start\n")
        
        FILE_PATH = f"../Data/interim/patient_genes_{GENE_FILE_VARIANT}.csv"
        df = pd.read_csv(FILE_PATH)
        
        ### Dataset split: training and test data, with SMOTE and without SMOTE
        X, y, X_train, X_test, y_train, y_test, test_case_ids = split_data(df, "tnbc", True)

        model_unbalanced = getModel(modelName)
        model_cv_unbalanced = getModel(modelName)
        
        model_smote = getModel(modelName)
        model_cv_smote = getModel(modelName)
    
        # Get weights and weighted model
        weights = getDataSetWeights(y_train)
        model_weighted = getWeightedModel(modelName, weights)
        model_cv_weighted = getWeightedModel(modelName, weights)


        scaler = StandardScaler()
        print(f'\nApply Scaling - Start')
        X_trainScaled = scaler.fit_transform(X_train)
        X_testScaled  = scaler.transform(X_test) 
        print(f'Apply Scaling - End')
    
        print("\nApply Smote - Start")
        X_train_smoteScaled, Y_train_smote = apply_smote_to_train(X_trainScaled, y_train)
        print("Apply Smote - End")


    # Model Performance Test
        print('\n-- Model Performance on test data')
        # Model without SMOTE, unweighted
        y_pred, y_prob = run_model(model_unbalanced, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, False, modelName)
        print_evaluated_model_accuracy(' + Model without SMOTE, unweighted', y_test, y_pred)
    
        # Model with SMOTE
        y_pred_smote, y_prob_smote = run_model(model_smote, X_train_smoteScaled, X_testScaled, Y_train_smote, y_test, test_case_ids, True, False, modelName)
        print_evaluated_model_accuracy(' + Model with SMOTE', y_test, y_pred_smote)
        
        # Model Weighted
        y_pred_weighted, y_prob_weighted = run_model(model_weighted, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, True, modelName)
        print_evaluated_model_accuracy(' + Model Weighted', y_test, y_pred_weighted)
        


    # Model Validation using CV
        print('\n-- Model Validation using 5-fold CV - Start')
        # Get metrics without SMOTE and unweighted
        print('\nwithout SMOTE and unweighted')
        metrics = run_cross_validation(model_cv_unbalanced, X_train, y_train, y_test, y_pred, y_prob, False, False, modelName)
        
        # Get metrics with SMOTE
        print('\nSMOTE')
        metrics_smote = run_cross_validation(model_cv_smote, X_train, y_train, y_test, y_pred_smote, y_prob_smote, True, False, modelName)
    
        # Get metrics Weighted
        print('\nWeighted')
        metrics_weighted = run_cross_validation(model_cv_weighted, X_train, y_train, y_test, y_pred_weighted, y_prob_weighted, False, True, modelName)
        
        print('\n-- Model Validation using 5-fold CV - End')


        print(f"{counter}/{total} - FeatureSet {GENE_FILE_VARIANT} - End")
        counter += 1
    
    print(f'*** - Applying {modelName} to features - End')

NameError: name 'ModelVariant' is not defined

# PCA

In [ ]:
for modelName in ModelVariant.__members__:

    print('\n\n')
    print(f'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII')
    print(f'*** - Applying {modelName} to features - Start')
    print(f'IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII\n')

    GENE_FILE_VARIANT = FeatureVariant.AUTOMATED

    FILE_PATH_TRAIN = f"../Data/interim/patient_genes_automated_train.csv"
    FILE_PATH_TEST = f"../Data/interim/patient_genes_automated_test.csv"
    
    df_train = pd.read_csv(FILE_PATH_TRAIN)
    df_test = pd.read_csv(FILE_PATH_TEST)
        
    ### Dataset split: training and test data, with SMOTE and without SMOTE
    X_train = df_train.drop(columns=["tnbc"])
    y_train = df_train['tnbc']

    X_test = df_test.drop(columns=["tnbc", "case_id"])
    y_test = df_test['tnbc']
    test_case_ids = df_test["case_id"]

    model_unbalanced = getModel(modelName)
    model_cv_unbalanced = getModel(modelName)
    
    model_smote = getModel(modelName)
    model_cv_smote = getModel(modelName)

    # Get weights and weighted model
    weights = getDataSetWeights(y_train)
    model_weighted = getWeightedModel(modelName, weights)
    model_cv_weighted = getWeightedModel(modelName, weights)


    scaler = StandardScaler()
    print(f'Apply Scaling - Start')
    X_trainScaled = scaler.fit_transform(X_train)
    X_testScaled  = scaler.transform(X_test) 
    print(f'Apply Scaling - End')

    print("\nApply Smote - Start")
    X_train_smoteScaled, Y_train_smote = apply_smote_to_train(X_trainScaled, y_train)
    print("\nApply Smote - End")

# Model Performance Test
    print('\n-- Model Performance on test data')
    
    # Model without SMOTE, unweighted
    y_pred, y_prob = run_model(model_unbalanced, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, False, modelName)
    print_evaluated_model_accuracy('+ Model without SMOTE, unweighted', y_test, y_pred)

    # Model with SMOTE
    y_pred_smote, y_prob_smote = run_model(model_smote, X_train_smoteScaled, X_testScaled, Y_train_smote, y_test, test_case_ids, True, False, modelName)
    print_evaluated_model_accuracy(' + Model with SMOTE', y_test, y_pred_smote)
    
    # Model Weighted
    y_pred_weighted, y_prob_weighted = run_model(model_weighted, X_trainScaled, X_testScaled, y_train, y_test, test_case_ids, False, True, modelName)
    print_evaluated_model_accuracy(' + Model Weighted', y_test, y_pred_weighted)
    


# Model Validation using CV
    print('\n-- Model Validation using 5-fold CV - Start')

    # Get metrics without SMOTE and unweighted
    print('\nwithout SMOTE and unweighted')
    metrics = run_cross_validation(model_cv_unbalanced, X_train, y_train, y_test, y_pred, y_prob, False, False, modelName)
    
    # Get metrics with SMOTE
    print('\nSMOTE')
    metrics_smote = run_cross_validation(model_cv_smote, X_train, y_train, y_test, y_pred_smote, y_prob_smote, True, False, modelName)

    # Get metrics Weighted
    print('\nWeighted')
    metrics_weighted = run_cross_validation(model_cv_weighted, X_train, y_train, y_test, y_pred_weighted, y_prob_weighted, False, True, modelName)


    print(f'*** - Applying {modelName} to features - End')